# Push to Chinook to Hugging face

We'll convert the tables in `chinook` to parquet files in a directory and then push the directory to Hugging Face.


## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [ ]:
import sqlite3 as db
import pandas as pd
import contextlib
import requests
from google.colab import userdata
import os
from huggingface_hub import HfApi, hf_hub_url
import duckdb


In [ ]:
%%bash
curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip


In [ ]:
!unzip -u chinook.zip


In [ ]:
!ls -l


Verify sqlite works.

In [ ]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)

In [ ]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  tables = pd.read_sql_query( query , db_con)

tables


## Read each table and convert to parquet


In [ ]:
# create target folder
!mkdir -p chinook


In [ ]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  for table in tables["name"]:
    print(table)
    query = f'''
      select * from {table} ;
    '''
    df = pd.read_sql_query( query, db_con)
    display(df.shape)
    output_path = f'./chinook/{table}.parquet'
    df.to_parquet(output_path, index=False)


In [ ]:
# verify
!ls -l chinook


## Read from parquet



In [ ]:
files = !ls -1 chinook/*.parquet
files


In [ ]:
for table in files:
  print(table)
  df = pd.read_parquet( table )
  display(df.shape)



## Authenticate to Hugging Face


In [ ]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

In [ ]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


In [ ]:
# set up account name
hf_account = os.environ["hf_account"] = userdata.get('hf_account')
hf_account


In [ ]:
# set up repo name
hf_repo = os.environ["hf_repo"] = "Data_Science-21"
hf_repo


In [ ]:
# set up file
hf_parquet = os.environ["hf_parquet"] = 'chinook'
hf_parquet


## Push folder with parquet files to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

# uncomment echo to actually push to Hugging Face
echo \
hf upload \
  --type dataset \
  ${hf_account}/${hf_repo} \
  ./${hf_parquet}/ \
  ${hf_parquet}/


In [ ]:
print(hf_upload.stdout)


## Query for parquet files on Hugging Face


In [ ]:
api = HfApi()
api


In [ ]:
REPO_ID = f"{hf_account}/{hf_repo}"
FOLDER_NAME = f"{hf_parquet}"
REPO_ID, FOLDER_NAME


In [ ]:
# List all files inside the dataset repository
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
repo_files


In [ ]:
# Filter for files inside your folder that end with .parquet
parquet_paths = [
  f"hf://datasets/{REPO_ID}/{file}"
    for file in repo_files
      if file.startswith(f"{FOLDER_NAME}/") and file.endswith(".parquet")
]

parquet_paths


In [ ]:
https_urls = [
    hf_hub_url(repo_id=REPO_ID, filename=file, repo_type="dataset")
    for file in repo_files
    if file.startswith(f"{FOLDER_NAME}/") and file.endswith(".parquet")
]

https_urls


## Read parquet from Hugging Face


In [ ]:
hf_url =  https_urls[0]
hf_url


In [ ]:
df3 = pd.read_parquet( hf_url )
df3.shape


In [ ]:
df3.iloc[:,:5].info()


## Create a data catalog DuckDb file


In [ ]:
queries = [
  f'''CREATE OR REPLACE VIEW {FOLDER_NAME}.{table}
    AS SELECT * FROM '{parquet}' ; '''
    for parquet in parquet_paths
      for table in [ parquet.split("/")[-1].split(".")[0] ]
]

print("\n".join(queries))


In [ ]:
catalog_db = f'{FOLDER_NAME}-hf.duckdb'
catalog_db


In [ ]:
con = duckdb.connect( catalog_db )
con.execute(f"CREATE SCHEMA IF NOT EXISTS {FOLDER_NAME};")
for query in queries:
  con.execute( query )
con.close()


In [ ]:
!ls -l


## Query the data catalog


In [ ]:
con = duckdb.connect( catalog_db )
df = con.execute(f"SELECT * FROM {FOLDER_NAME}.employees ;").df()
con.close()
df

In [ ]:
con = duckdb.connect( catalog_db )
df = con.execute("show all tables").df()
con.close()
df


In [ ]:
!ls -l


## Push data catalog to Hugging Face


In [ ]:
# set up file
hf_catalog_db = os.environ["hf_catalog_db"] = catalog_db
hf_catalog_db


In [ ]:
%%capture hf_upload
%%bash

# uncomment echo to actually push to Hugging Face
echo \
hf upload \
  --type dataset \
  ${hf_account}/${hf_repo} \
  ./${hf_catalog_db}


In [ ]:
print(hf_upload.stdout)


## Verify data catalog on Hugging Face


In [ ]:
CATALOG_FILE = hf_catalog_db
REPO_ID, CATALOG_FILE


In [ ]:
# 1. Get the direct, routable HTTPS download link for the file
remote_db_url = hf_hub_url(repo_id=REPO_ID, filename=CATALOG_FILE, repo_type="dataset")
remote_db_url


In [ ]:
con = duckdb.connect()
con.execute(f"ATTACH '{remote_db_url}' AS hf (READ_ONLY);")


In [ ]:
df = con.execute("SELECT * FROM hf.chinook.albums LIMIT 5").df()
df


In [ ]:
con.query( "show all tables ").df()
